In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
from urls import urls

import re
import json

from pymongo import MongoClient
import os
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableMap, RunnableLambda
from langchain_anthropic import ChatAnthropic
from langchain.memory import ConversationBufferMemory
from langchain_core.output_parsers import StrOutputParser

from langchain_groq import ChatGroq

from typing import Literal, Dict
from typing import Literal, Dict, Optional
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field



class FeatureDetails(BaseModel):
    rating: Literal["good", "bad", "neutral"]
    details: str

class InsurancePolicy(BaseModel):
    insurance_name: str
    provider: str
    features: Dict[str, FeatureDetails] = Field(
        description="Dictionary of features and their ratings/details"
    )

parser = PydanticOutputParser(pydantic_object=InsurancePolicy)
format_instructions = parser.get_format_instructions()

load_dotenv()

True

In [2]:
def scrape_icici_lombard(url):
    # URL to scrape
    # Configure Chrome options
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')
    
    # Initialize the browser
    browser = webdriver.Chrome(options=options)
    
    try:
        # Access the website
        browser.get(url)
        time.sleep(5)
        
        # Get page source
        html_content = browser.page_source
        
        # Parse HTML with BeautifulSoup
        soup = BeautifulSoup(html_content, 'html.parser')
        
        # Extract all text from the page
        all_text = soup.get_text(separator=' ', strip=True)

        print(f"Successfully scraped {len(all_text)} characters of text")
        
        return all_text
        
    except Exception as e:
        print(f"Error: {str(e)}")
        return None
    finally:
        # Close the browser
        browser.quit()

In [57]:
mixtral = 'mixtral-8x7b-32768'
llama = 'llama3-70b-8192'

model = ChatGroq(temperature=0, model_name=llama) 

In [31]:
t = scrape_icici_lombard('https://joinditto.in/health-insurance/acko/platinum-health')

Starting browser...
Accessing https://joinditto.in/health-insurance/acko/platinum-health
Waiting for page to load...
Successfully scraped 6890 characters of text
Browser closed


In [ ]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        Based on the text given by human, please format the text in below format without any extra verbose. There should be no explanation in the output.
From the text, get the data related to below features and json format the data as shown.
Co-payment  
Room-rent-limit  
Disease-sub-limit  
Pre-Post-hospitalization 
critical-illness 
Low-Waiting-period  
Daycare-Treatment-cover  
Restoration-Benefit  
maternity-coverage
no-claim-bonus
     
       
FORMAT:
insurance_name : <policy name>  <should be in lowercase and words seperated by ->
provider : <company name> <should be in lowercase and words seperated by ->
description: <description of policy> i need the proper description of the policy on what the policy is about
features :
     co-payment:
      rating: <good/bad/neutral>
      details: <details about why its good/bad/neutral>
     
     Room-rent-limit:
      rating: <good/bad/neutral>
      details: <details about why its good/bad/neutral>
    and so on for all the features mentioned above. If detail about certain feature is not there just mention as NA in details and in rating.

     Please give output as per below format:  
{format_instructions}
"""),
    ("human", "{question}")
])


chain = (
    RunnableMap({
        "question": lambda x: x["question"],
        "format_instructions": lambda x: format_instructions
    })
    | template 
    | model 
    | StrOutputParser()
)

In [59]:

def extract_json_from_text(text):
    # First try to find JSON within triple backticks
    backtick_pattern = r'```\s*([\s\S]*?)\s*```'
    backtick_matches = re.findall(backtick_pattern, text)
    
    if backtick_matches:
        # Try each match from triple backticks
        for match in backtick_matches:
            try:
                # Look for JSON pattern within the match
                json_pattern = r'(\{[\s\S]*\})'
                json_matches = re.findall(json_pattern, match)
                if json_matches:
                    for json_str in json_matches:
                        try:
                            return json.loads(json_str)
                        except json.JSONDecodeError:
                            continue
            except Exception:
                continue
    
    # If no valid JSON found in backticks, try to find JSON directly in the text
    json_pattern = r'(\{[\s\S]*?\})'
    potential_jsons = re.findall(json_pattern, text)
    
    # Try each potential JSON string, starting with the longest ones
    # (more likely to be complete JSON objects)
    potential_jsons.sort(key=len, reverse=True)
    
    for json_str in potential_jsons:
        try:
            # Check if this is a valid, non-empty JSON object
            parsed = json.loads(json_str)
            if isinstance(parsed, dict) and parsed:
                return parsed
        except json.JSONDecodeError:
            continue
    
    raise ValueError("No valid JSON found in the text")

In [3]:
client = MongoClient("mongodb://localhost:27017/")
db = client["insurance_db"]
collection = db["health_insurance"]

In [ ]:
#client.drop_database("insurance_db")

In [62]:
cnt=0
import time
for name in policy_names:
    print("scrapping url", name)
    t = scrape_icici_lombard(name)
    if '404 - Page Not Found' in t:
        continue
    print("exatracting using llm")
    response = chain.invoke({"question": t})
    structured_output = extract_json_from_text(response)
    collection.insert_one(structured_output)
    
    print("======================================================================", cnt)
    cnt+=1
    if cnt%5==0:
        time.sleep(4)

scrapping url https://joinditto.in/health-insurance/acko/platinum-health
Successfully scraped 6890 characters of text
exatracting using llm
====================================================================== 0
scrapping url https://joinditto.in/health-insurance/acko/standard-health
Successfully scraped 7102 characters of text
exatracting using llm
====================================================================== 1
scrapping url https://joinditto.in/health-insurance/acko/aditya-birla-health-insurance
Successfully scraped 2153 characters of text
scrapping url https://joinditto.in/health-insurance/acko/zuno-health-insurance
Successfully scraped 2153 characters of text
scrapping url https://joinditto.in/health-insurance/acko/hdfc-ergo-health-insurance
Successfully scraped 2153 characters of text
scrapping url https://joinditto.in/health-insurance/apollo-munich/easy-health-exclusive
Successfully scraped 8427 characters of text
exatracting using llm
==================================

In [69]:
structured_output

{'insurance_name': 'united-india-individual-gold-plan',
 'provider': 'united-india',
 'description': 'United India Individual Gold Plan is a comprehensive health insurance policy that covers various medical expenses, including hospitalization, day care treatments, and alternative medicine.',
 'features': {'co-payment': {'rating': 'good',
   'details': 'No co-payment, the insurer will bear the entire cost of treatment up to the sum insured.'},
  'Room-rent-limit': {'rating': 'bad',
   'details': 'Room rent limit of 1% of the sum insured for normal rooms and 2% for ICU.'},
  'Disease-sub-limit': {'rating': 'bad',
   'details': 'Disease-wise sub-limits for certain diseases like Cataracts, Hernia, Hysterectomy, etc.'},
  'Pre-Post-hospitalization': {'rating': 'good',
   'details': 'Pre and post hospitalization expenses covered up to 10% of sum insured for 30 days before and 60 days after discharge.'},
  'critical-illness': {'rating': 'NA', 'details': 'NA'},
  'Low-Waiting-period': {'rating

### List of companies

In [4]:
# No. of unique providers

unique_providers = collection.distinct("provider")
# Count total unique providers
total_providers = len(unique_providers)
total_providers

18

In [5]:
unique_providers

['acko',
 'aditya-birla',
 'bajaj-allianz',
 'bharti-axa',
 'care',
 'hdfc-ergo',
 'icici-lombard',
 'iffco-tokio',
 'manipal-cigna',
 'new-india-assurance',
 'niva-bupa-erstwhile-max-bupa',
 'oriental-insurance',
 'reliance-general',
 'royal-sundaram',
 'sbi',
 'star-health',
 'tata-aig',
 'united-india']

### List of plans under each provider

In [66]:
pipeline = [
    {"$group": {"_id": "$provider", "insurance_names": {"$addToSet": "$insurance_name"}}}
]

# Execute the query
result = collection.aggregate(pipeline)

# Convert result to dictionary
provider_insurances = {entry["_id"]: entry["insurance_names"] for entry in result}

# Print the output
print(provider_insurances)

{'tata-aig': ['tata-aig-medicare-plan', 'tata-aig-health-supercharge-plan', 'tata-aig-elder-care-plan', 'tata-aig-medicare-plus-super-top-up'], 'aditya-birla': ['aditya-birla-activ-one-vip', 'aditya-birla-activ-one-savr', 'aditya-birla-activ-one-vytl', 'aditya-birla-activ-fit-preferred', 'aditya-birla-activ-one-nxt'], 'hdfc-ergo': ['hdfc-ergo-optima-secure-plan', 'hdfc-ergo-optima-restore', 'hdfc-ergo-optima-secure-global', 'hdfc-ergo-optima-secure-global-plus', 'hdfc-ergo-easy-health-premium-plan', 'hdfc-ergo-optima-super-secure-plan', 'hdfc-ergo-easy-health-exclusive-plan', 'hdfc-ergo-optima-lite-plan', 'hdfc-ergo-myhealth-suraksha-platinum'], 'oriental-insurance': ['health-of-privileged-elders', 'oriental-insurance-mediclaim'], 'care': ['care-supreme-senior-super'], 'bajaj-allianz': ['bajaj-allianz-health-guard-platinum', 'bajaj-allianz-health-guard-gold', 'bajaj-allianz-health-ensure-family-plan', 'bajaj-allianz-silver-health', 'bajaj-allianz-health-guard-silver'], 'bharti-axa': ['

In [67]:
for k,v in provider_insurances.items():
    print(k)
    for i in v:
        print(i)
    print("=========================================================")

tata-aig
tata-aig-medicare-plan
tata-aig-health-supercharge-plan
tata-aig-elder-care-plan
tata-aig-medicare-plus-super-top-up
aditya-birla
aditya-birla-activ-one-vip
aditya-birla-activ-one-savr
aditya-birla-activ-one-vytl
aditya-birla-activ-fit-preferred
aditya-birla-activ-one-nxt
hdfc-ergo
hdfc-ergo-optima-secure-plan
hdfc-ergo-optima-restore
hdfc-ergo-optima-secure-global
hdfc-ergo-optima-secure-global-plus
hdfc-ergo-easy-health-premium-plan
hdfc-ergo-optima-super-secure-plan
hdfc-ergo-easy-health-exclusive-plan
hdfc-ergo-optima-lite-plan
hdfc-ergo-myhealth-suraksha-platinum
oriental-insurance
health-of-privileged-elders
oriental-insurance-mediclaim
care
care-supreme-senior-super
bajaj-allianz
bajaj-allianz-health-guard-platinum
bajaj-allianz-health-guard-gold
bajaj-allianz-health-ensure-family-plan
bajaj-allianz-silver-health
bajaj-allianz-health-guard-silver
bharti-axa
bharti-axa-smart-super-health-assure
bharti-axa-smart-super-health
bharti-axa-smart-super-top-up
royal-sundaram
ro

In [68]:
provider_name = "united-india"  # Change to the provider you want to analyze

# Query to get insurance policies for a specific provider
policies = collection.find({"provider": provider_name}, {"insurance_name": 1, "features": 1, "_id": 0})

# Dictionary to store good and bad features for each insurance plan
provider_analysis = {}

for policy in policies:
    insurance_name = policy["insurance_name"]
    features = policy["features"]

    good_features = []
    bad_features = []

    for feature, details in features.items():
        if details["rating"].lower() == "good":
            good_features.append(feature)
        elif details["rating"].lower() == "bad":
            bad_features.append(feature)

    provider_analysis[insurance_name] = {
        "good_features": good_features,
        "bad_features": bad_features
    }

# Print the analysis
print(provider_analysis)

{'united-india-senior-citizen-plan': {'good_features': ['co-payment', 'Pre-Post-hospitalization', 'Daycare-Treatment-cover'], 'bad_features': ['Room-rent-limit', 'Disease-sub-limit', 'Low-Waiting-period', 'Restoration-Benefit', 'no-claim-bonus']}, 'united-india-family-medicare-plan': {'good_features': ['co-payment', 'Pre-Post-hospitalization', 'Daycare-Treatment-cover', 'Restoration-Benefit', 'no-claim-bonus'], 'bad_features': ['Disease-sub-limit']}, 'united-india-individual-gold-plan': {'good_features': ['co-payment', 'Pre-Post-hospitalization', 'Daycare-Treatment-cover'], 'bad_features': ['Room-rent-limit', 'Disease-sub-limit', 'Restoration-Benefit', 'no-claim-bonus']}}
